# توليد فن مرتبط بالمشاعر + تحليل (Notebook)

نفس منطق تطبيق Streamlit: إدخال **مشاعر كنص** → توليد صورة (Stable Diffusion) → تحليل JSON (Gemini أو Claude).

**شغّل الخلايا من الأعلى للأسفل.** افتح المجلد `d:\depi project` كـ working directory، أو ضع هذا الملف في جذر المشروع وشغّل من هناك.

المفاتيح في ملف `.env` (مثل `GEMINI_API_KEY`, `HF_TOKEN`) أو عيّنها يدويًا في الخلية أدناه.

In [1]:
# مسار المشروع + تحميل .env
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "emotion_art").is_dir():
    raise RuntimeError(
        "شغّل الـ notebook من مجلد المشروع (حيث يوجد emotion_art). "
        f"المجلد الحالي: {ROOT}"
    )

sys.path.insert(0, str(ROOT))

def _load_env(path: Path) -> None:
    if not path.is_file():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        k, v = k.strip(), v.strip().strip('"').strip("'")
        if k and k not in os.environ:
            os.environ[k] = v

_load_env(ROOT / ".env")
print("Project:", ROOT)

Project: D:\depi project


## إعداداتك

- `USER_EMOTION`: نص المشاعر (مثل `happy`).
- `ANALYSIS_PROVIDER`: `"gemini"` (مجاني تقريبًا) أو `"anthropic"`.
- المفاتيح: من `.env` أو عيّن المتغيرات هنا قبل تشغيل الخلايا التالية.

In [2]:
USER_EMOTION = "happy"  # غيّرها كما تريد
ANALYSIS_PROVIDER = "gemini"  # أو "anthropic"

# اختياري: تجاوز .env — فعّل السطر المناسب
# os.environ["GEMINI_API_KEY"] = "..."
# os.environ["ANTHROPIC_API_KEY"] = "..."
# os.environ["HF_TOKEN"] = "hf_..."

SEED = 42
STEPS = 30
GUIDANCE = 7.5

In [ ]:
import json
import matplotlib.pyplot as plt

from emotion_art.generate import generate_artwork, get_pipeline
from emotion_art.analyze import analyze_artwork_json, build_result_payload
from emotion_art.io_utils import save_image_png
from emotion_art.visualizations import (
    color_emphasis_overlay,
    highlight_faces,
    word_cloud_image,
    analysis_text_for_wordcloud,
)

print("CUDA:", __import__("torch").cuda.is_available())

c:\Program Files\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.10.0+cpu)
    Python  3.10.11 (you have 3.13.5)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


## 1) تحميل نموذج الرسم (أول مرة قد يستغرق تنزيلًا طويلًا)

In [ ]:
print("Loading Stable Diffusion pipeline (first run downloads several GB)...")
get_pipeline()
print("Pipeline ready.")

## 2) توليد الصورة

In [ ]:
image = generate_artwork(
    USER_EMOTION,
    num_inference_steps=STEPS,
    guidance_scale=GUIDANCE,
    seed=SEED,
)
out_path = save_image_png(image, prefix="notebook_artwork")
print("Saved:", out_path)

plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis("off")
plt.title(USER_EMOTION)
plt.tight_layout()
plt.show()

## 3) التحليل (JSON)

In [ ]:
if ANALYSIS_PROVIDER == "anthropic":
    key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
else:
    key = (os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY") or "").strip()

if not key:
    raise RuntimeError("ضع GEMINI_API_KEY أو ANTHROPIC_API_KEY في .env أو في os.environ")

analysis = analyze_artwork_json(
    image,
    USER_EMOTION,
    api_key=key,
    provider=ANALYSIS_PROVIDER,
)
payload = build_result_payload(USER_EMOTION, out_path, analysis)
print(json.dumps(payload, indent=2, ensure_ascii=False))

## 4) اختياري: خرائط لونية + وجوه + سحابة كلمات

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(image)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(color_emphasis_overlay(image))
axes[1].set_title("Color emphasis overlay")
axes[1].axis("off")
plt.tight_layout()
plt.show()

face_img, n = highlight_faces(image)
plt.figure(figsize=(6, 6))
plt.imshow(face_img)
plt.title(f"Face highlights (count={n})")
plt.axis("off")
plt.show()

wc = word_cloud_image(analysis_text_for_wordcloud(analysis))
if wc:
    plt.figure(figsize=(10, 5))
    plt.imshow(wc)
    plt.axis("off")
    plt.title("Word cloud")
    plt.show()